## Data Cleaning & Transformation

### Creating a spark session for data interaction

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('Olist Data Cleaning & Transformation').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 21:47:45 INFO SparkEnv: Registering MapOutputTracker
26/06/26 21:47:46 INFO SparkEnv: Registering BlockManagerMaster
26/06/26 21:47:46 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/06/26 21:47:46 INFO SparkEnv: Registering OutputCommitCoordinator


In [2]:
hdfs_base_path = '/user/Marho/project/olist/data/'

options = {
    'header': True,
    'inferSchema': True
}
 
productSchema = '''
    product_id STRING,
    product_category_name STRING,
    product_name_lenght DOUBLE,
    product_description_lenght DOUBLE,
    product_photos_qty DOUBLE,
    product_weight_g DOUBLE,
    product_length_cm DOUBLE,
    product_height_cm DOUBLE,
    product_width_cm DOUBLE
'''

customer_df = spark.read.options(**options).csv(hdfs_base_path+'olist_customers_dataset.csv')
geolocation_df = spark.read.options(**options).csv(hdfs_base_path+ 'olist_geolocation_dataset.csv')
order_items_df = spark.read.options(**options).csv(hdfs_base_path+'olist_order_items_dataset.csv')
payment_df = spark.read.options(**options).csv(hdfs_base_path+'olist_order_payments_dataset.csv')
review_df = spark.read.options(**options).csv(hdfs_base_path+'olist_order_reviews_dataset.csv')
orders_df = spark.read.options(**options).csv(hdfs_base_path+'olist_orders_dataset.csv')
products_df = spark.read.option('header',options['header']).schema(productSchema).csv(hdfs_base_path+'olist_products_dataset.csv')
sellers_df = spark.read.options(**options).csv(hdfs_base_path+'olist_sellers_dataset.csv')
products_category_df = spark.read.options(**options).csv(hdfs_base_path+'product_category_name_translation.csv')

_Helper Functions_

In [3]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Discovering Missing Values

In [4]:
def get_missing_values(df, df_name):
    print(f'Missing Values for {df_name}')
    return df.select([count(when(col(c).isNull(),1)).alias(c) for c in df.columns]).show()

#### Customers Data

In [5]:
get_missing_values(customer_df, 'Customer Dataframe')

Missing Values for Customer Dataframe


+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



#### Order Data

In [6]:
get_missing_values(orders_df, 'orders')

Missing Values for orders


+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



#### Order Item

In [7]:
get_missing_values(order_items_df, 'Order Items')

Missing Values for Order Items
+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|       0|            0|         0|        0|                  0|    0|            0|
+--------+-------------+----------+---------+-------------------+-----+-------------+



#### Payment

In [8]:
get_missing_values(payment_df, 'payment')

Missing Values for payment
+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|       0|                 0|           0|                   0|            0|
+--------+------------------+------------+--------------------+-------------+



####  Geolocation

In [9]:
get_missing_values(geolocation_df,'Geolocation')

Missing Values for Geolocation


+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+



#### Review Data

In [10]:
get_missing_values(review_df, 'Review')

Missing Values for Review


+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        1|    2236|        2380|               92157|                 63079|                8764|                   8785|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



#### Products Data

In [11]:
get_missing_values(products_df, 'Products')

Missing Values for Products
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|         0|                  610|                610|                       610|               610|               2|                2|                2|               2|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



#### Sellers Data

In [12]:
get_missing_values(sellers_df, 'Sellers')

Missing Values for Sellers
+---------+----------------------+-----------+------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+----------------------+-----------+------------+
|        0|                     0|          0|           0|
+---------+----------------------+-----------+------------+



#### Category Translation Data

In [13]:
get_missing_values(products_category_df, 'Category Translation')

Missing Values for Category Translation
+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|                    0|                            0|
+---------------------+-----------------------------+



#### Nulls Analysis

These dataframes had some null values:

*   Order Dataframe
*   Review data
*   Product data

### Handling Missing Values

#### Order Dataframe

Missing Columns:

* order_approved_at
* order_delivered_carrier_date
* order_delivered_customer_date

In [14]:
all_orders = (orders_df.select(['order_id','order_status',
                                'order_purchase_timestamp',
                                'order_approved_at', 
                                'order_delivered_carrier_date', 
                                'order_delivered_customer_date']))
deliveredOrder = all_orders.filter(col('order_status' ) == lit('delivered'))
deliveredOrder.show(truncate=False)

+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+
|order_id                        |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|
+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|delivered   |2017-10-02 10:56:33     |2017-10-02 11:07:15|2017-10-04 19:55:00         |2017-10-10 21:25:13          |
|53cdb2fc8bc7dce0b6741e2150273451|delivered   |2018-07-24 20:41:37     |2018-07-26 03:24:27|2018-07-26 14:31:00         |2018-08-07 15:27:45          |
|47770eb9100c2d0c44946d9cf07ec65d|delivered   |2018-08-08 08:38:49     |2018-08-08 08:55:23|2018-08-08 13:50:00         |2018-08-17 18:06:29          |
|949d5b44dbf5de918fe9c16f97b45f8a|delivered   |2017-11-18 19:28:06     |2017-11-18 19:45

In [15]:
get_missing_values(deliveredOrder, 'delivered orders')

Missing Values for delivered orders
+--------+------------+------------------------+-----------------+----------------------------+-----------------------------+
|order_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|
+--------+------------+------------------------+-----------------+----------------------------+-----------------------------+
|       0|           0|                       0|               14|                           2|                            8|
+--------+------------+------------------------+-----------------+----------------------------+-----------------------------+



In [16]:
invoicedOrder = all_orders.filter(col('order_status' ) == lit('invoiced'))
invoicedOrder.show(truncate=False)

+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+
|order_id                        |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|
+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+
|136cce7faa42fdb2cefd53fdc79a6098|invoiced    |2017-04-11 12:22:08     |2017-04-13 13:25:17|NULL                        |NULL                         |
|0760a852e4e9d89eb77bf631eaaf1c84|invoiced    |2018-08-03 17:44:42     |2018-08-07 06:15:14|NULL                        |NULL                         |
|38b7efdf33dd5561f4f5d4f6e07b0414|invoiced    |2017-08-01 18:17:41     |2017-08-01 18:32:30|NULL                        |NULL                         |
|51b0dccc8596ce37a930dff2d63a10a2|invoiced    |2017-05-05 22:34:48     |2017-05-05 22:45

In [17]:
get_missing_values(invoicedOrder, 'invoiced')

Missing Values for invoiced
+--------+------------+------------------------+-----------------+----------------------------+-----------------------------+
|order_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|
+--------+------------+------------------------+-----------------+----------------------------+-----------------------------+
|       0|           0|                       0|                0|                         314|                          314|
+--------+------------+------------------------+-----------------+----------------------------+-----------------------------+



In [18]:
canceledOrder = all_orders.filter(col('order_status' ) == lit('canceled'))
canceledOrder.show(truncate=False)

+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+
|order_id                        |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|
+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+
|1b9ecfe83cdc259250e1a8aca174f0ad|canceled    |2018-08-04 14:29:27     |2018-08-07 04:10:26|NULL                        |NULL                         |
|714fb133a6730ab81fa1d3c1b2007291|canceled    |2018-01-26 21:34:08     |2018-01-26 21:58:39|2018-01-29 22:33:25         |NULL                         |
|3a129877493c8189c59c60eb71d97c29|canceled    |2018-01-25 13:34:24     |2018-01-25 13:50:20|2018-01-26 21:42:18         |NULL                         |
|00b1cb0320190ca0daa2c88b35206009|canceled    |2018-08-28 15:26:39     |NULL            

In [19]:
get_missing_values(canceledOrder, 'canceled')

Missing Values for canceled
+--------+------------+------------------------+-----------------+----------------------------+-----------------------------+
|order_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|
+--------+------------+------------------------+-----------------+----------------------------+-----------------------------+
|       0|           0|                       0|              141|                         550|                          619|
+--------+------------+------------------------+-----------------+----------------------------+-----------------------------+



**NOTE: _The missing dates field varies depending on order status_**
**_So we will handle just the missing dates for the delivered orders_**

In [20]:
# Handle missing date fields for delivered orders only
# Order matters: fix approved_at → customer_date → carrier_date (midpoint of the two)

orders_df_cleaned = (
    orders_df.withColumn(
        'order_approved_at',
        when(
            (col('order_status') == lit('delivered')) &
            (col('order_approved_at').isNull()) &
            (col('order_purchase_timestamp').isNotNull()),
            col('order_purchase_timestamp')
        ).otherwise(col('order_approved_at'))
    ).withColumn(
        'order_delivered_customer_date',
        when(
            (col('order_status') == lit('delivered')) &
            (col('order_delivered_customer_date').isNull()) &
            (col('order_estimated_delivery_date').isNotNull()),
            col('order_estimated_delivery_date')
        ).otherwise(col('order_delivered_customer_date'))
    ).withColumn(
        'order_delivered_carrier_date',
        when(
            (col('order_status') == lit('delivered')) &
            (col('order_delivered_carrier_date').isNull()) &
            (col('order_approved_at').isNotNull()) &
            (col('order_delivered_customer_date').isNotNull()),
            to_timestamp(
                (unix_timestamp('order_approved_at') + unix_timestamp('order_delivered_customer_date')) / 2
            )
        ).otherwise(col('order_delivered_carrier_date'))
    )
)

#### Product Dataframe

Missing Columns:

* product_category_name
* product_name_lenght
* product_description_lenght
* product_photos_qty
* product_weight_g
* product_length_cm
* product_height_cm 
* product_width_cm

In [21]:
products_df.show()

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|               40.0|                     287.0|               1.0|           225.0|             16.0|             10.0|            14.0|
|3aa071139cb16b67c...|                artes|               44.0|                     276.0|               1.0|          1000.0|             30.0|             18.0|            20.0|
|96bd76ec8810374ed...|        esporte_lazer|               46.0|                     250.0|    

In [22]:
get_missing_values(products_df,'products')

Missing Values for products
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|         0|                  610|                610|                       610|               610|               2|                2|                2|               2|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



**NOTE: _The fields: product_category_name, product_name_lenght, product_description_lenght & product_photos_qty_**
**_would be ignored as they would need to be determined from the source. However the  fields: product_weight_g,  product_length_cm, product_height_cm and product_width_cm, can be computed using Impute class as their values are quite small_**

##### Impute Missing Values

In [23]:
# Impute 2 missing physical dimension values using mode, then replace originals with imputed columns


from pyspark.ml.feature import Imputer


imputer = Imputer(inputCols=['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'],
                 outputCols=['product_weight_g_imp', 'product_length_cm_imp', 'product_height_cm_imp', 'product_width_cm_imp']).setStrategy('mode')

products_df_cleaned = (
    imputer.fit(products_df).transform(products_df)
    .drop('product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm')
    .withColumnRenamed('product_weight_g_imp', 'product_weight_g')
    .withColumnRenamed('product_length_cm_imp', 'product_length_cm')
    .withColumnRenamed('product_height_cm_imp', 'product_height_cm')
    .withColumnRenamed('product_width_cm_imp', 'product_width_cm')
)

### Standardizing Schemas

In [24]:
def getSchema(df, df_name):
    print(f'Schema of {df_name}')
    df.printSchema()
    df.show(2)

#### Customers Data

In [25]:
getSchema(customer_df, 'customers')

Schema of customers
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 2 rows



#### Order Data

In [26]:
getSchema(orders_df_cleaned, 'orders')

Schema of orders
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+------

#### Order Item

In [27]:
getSchema(order_items_df, 'order items')

Schema of order items
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
+--------------------+-------------+--------------------+---------------

#### Payment

In [28]:
getSchema(payment_df, 'payment')

Schema of payment
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 2 rows



####  Geolocation

In [29]:
getSchema(geolocation_df, 'geolocation')

Schema of geolocation
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

+---------------------------+-------------------+------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|   geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+------------------+----------------+-----------------+
|                       1037| -23.54562128115268|-46.63929204800168|       sao paulo|               SP|
|                       1046|-23.546081127035535|-46.64482029837157|       sao paulo|               SP|
+---------------------------+-------------------+------------------+----------------+-----------------+
only showing top 2 rows



#### Review Data

In [30]:
getSchema(review_df, 'reviews')

Schema of reviews
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|                NULL|                  NULL| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|           

#### Products Data

In [31]:
getSchema(products_df_cleaned, 'products')

Schema of products
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: double (nullable = true)
 |-- product_description_lenght: double (nullable = true)
 |-- product_photos_qty: double (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+--

#### Sellers Data

In [32]:
getSchema(sellers_df, 'sellers')

Schema of sellers
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

+--------------------+----------------------+-----------+------------+
|           seller_id|seller_zip_code_prefix|seller_city|seller_state|
+--------------------+----------------------+-----------+------------+
|3442f8959a84dea7e...|                 13023|   campinas|          SP|
|d1b65fc7debc3361e...|                 13844| mogi guacu|          SP|
+--------------------+----------------------+-----------+------------+
only showing top 2 rows



#### Category Translation Data

In [33]:
getSchema(products_category_df, 'Category translation')

Schema of Category translation
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|         beleza_saude|                health_beauty|
| informatica_acess...|         computers_accesso...|
+---------------------+-----------------------------+
only showing top 2 rows



#### Schema Analysis

Customer's Schema has improper datatype within this column:

*   customer_zip_code_prefix

Geolocation's Schema has improper datatype within this column:

*   geolocation_zip_code_prefix

Review's Schema has improper datatype wiithin these columns:

*   review_score
*   review_creation_date
*   review_answer_timestamp

Sellers's Schema has improper datatype within this column:

*   seller_zip_code_prefix



### Handling Schema Values

#### Customer Dataframe

Faulty Columns:

*   customer_zip_code_prefix (int)

In [34]:
customers_df_cleaned = (customer_df.withColumn('customer_zip_code_prefix', col('customer_zip_code_prefix')
                                               .cast(StringType())))

getSchema(customers_df_cleaned, 'customers')

Schema of customers
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 2 rows



#### Geolocation Dataframe

Faulty Columns:

*   geolocation_zip_code_prefix (int)

In [35]:
geolocation_cleaned = (geolocation_df.withColumn('geolocation_zip_code_prefix', col('geolocation_zip_code_prefix')
                                               .cast(StringType())))

getSchema(geolocation_cleaned, 'geolocation')

Schema of geolocation
root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

+---------------------------+-------------------+------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|   geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+------------------+----------------+-----------------+
|                       1037| -23.54562128115268|-46.63929204800168|       sao paulo|               SP|
|                       1046|-23.546081127035535|-46.64482029837157|       sao paulo|               SP|
+---------------------------+-------------------+------------------+----------------+-----------------+
only showing top 2 rows



#### Review Dataframe

Faulty Columns:

*   review_score (string)
*   review_creation_date (string)
*   review_answer_timestamp (string)

In [36]:
review_cleaned_df = (review_df.withColumn('review_score', col('review_score').cast(IntegerType()))
                     .withColumn('review_creation_date', col('review_creation_date').cast(TimestampType()))
                     .withColumn('review_answer_timestamp', col('review_answer_timestamp').cast(TimestampType())))

In [37]:
getSchema(review_cleaned_df, 'review')

Schema of review
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|                NULL|                  NULL| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|     

#### Sellers Dataframe

Faulty Columns:

*   seller_zip_code_prefix (int)

In [38]:
sellers_df_cleaned = (sellers_df.withColumn('seller_zip_code_prefix', col('seller_zip_code_prefix')
                                               .cast(StringType())))

getSchema(sellers_df_cleaned, 'sellers')

Schema of sellers
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

+--------------------+----------------------+-----------+------------+
|           seller_id|seller_zip_code_prefix|seller_city|seller_state|
+--------------------+----------------------+-----------+------------+
|3442f8959a84dea7e...|                 13023|   campinas|          SP|
|d1b65fc7debc3361e...|                 13844| mogi guacu|          SP|
+--------------------+----------------------+-----------+------------+
only showing top 2 rows



### Standardizing Payment Types

In [39]:
payment_df.select('payment_type').distinct().show()

+------------+
|payment_type|
+------------+
| not_defined|
| credit_card|
|      boleto|
|  debit_card|
|     voucher|
+------------+



In [40]:
paymentDf_cleaned = (
    payment_df.withColumn(
        "payment_type",
        when(col("payment_type") == "credit_card", "credit card")
        .when(col("payment_type") == "boleto", "bank transfer")
        .when(col("payment_type") == "debit_card", "debit card")
        .when(col("payment_type") == "not_defined", "other")
        .otherwise(col("payment_type"))
    )
)

In [41]:
paymentDf_cleaned.select('payment_type').distinct().show()

+-------------+
| payment_type|
+-------------+
|  credit card|
|   debit card|
|        other|
|bank transfer|
|      voucher|
+-------------+



### Removing Duplicate Records

#### Customers Data

In [42]:
customers_df_cleaned = customers_df_cleaned.dropDuplicates()

#### Order Data

In [43]:
orders_df_cleaned = orders_df_cleaned.dropDuplicates()

#### Order Item

In [44]:
order_items_df_cleaned = order_items_df.dropDuplicates()

#### Payment

In [45]:
paymentDf_cleaned = paymentDf_cleaned.dropDuplicates()

####  Geolocation

In [46]:
geolocation_df_cleaned = geolocation_cleaned.dropDuplicates()

#### Review Data

In [47]:
review_cleaned_df = review_cleaned_df.dropDuplicates()

#### Products Data

In [48]:
products_df_cleaned = products_df_cleaned.dropDuplicates()

#### Sellers Data

In [49]:
sellers_df_cleaned = sellers_df_cleaned.dropDuplicates()

#### Category Translation Data

In [50]:
products_category_df_cleaned = products_category_df.dropDuplicates()

### Feature Engineering

#### Approval Time

In [51]:
orders_df_cleaned = (orders_df_cleaned.withColumn('approval_time', 
                                                  date_diff(col('order_purchase_timestamp'),
                                                            col('order_approved_at'))
                                                 ))

#### Carrier Handling Time

In [52]:
orders_df_cleaned = (orders_df_cleaned.withColumn('carrier_handling_time', 
                                                  date_diff(col('order_approved_at'),
                                                            col('order_delivered_carrier_date'))
                                                 ))

#### Delivery Delay

In [53]:
orders_df_cleaned = (orders_df_cleaned.withColumn('delivery_delay', 
                                                  date_diff(col('order_estimated_delivery_date'),
                                                            col('order_delivered_customer_date'))
                                                 ))

#### Delivery Duration

In [54]:
orders_df_cleaned = (orders_df_cleaned.withColumn('delivery_duration', 
                                                  date_diff(col('order_delivered_customer_date'),
                                                            col('order_purchase_timestamp'))
                                                 ))

#### Shipping Time

In [55]:
orders_df_cleaned = (orders_df_cleaned.withColumn('shipping_time', 
                                                  date_diff(col('order_delivered_carrier_date'),
                                                            col('order_delivered_customer_date'))
                                                 ))

#### Order Purchase Day

In [61]:
orders_df_cleaned = (orders_df_cleaned
                     .withColumn('purchase_day', day('order_purchase_timestamp'))
                     .withColumn('purchase_day_name', date_format('order_purchase_timestamp','EEE'))
                    )

#### Order Purchase Month

In [62]:
orders_df_cleaned = (orders_df_cleaned
                     .withColumn('purchase_month', month('order_purchase_timestamp'))
                     .withColumn('purchase_month_name', date_format('order_purchase_timestamp', 'MMM'))
                    )

#### Order Purchase Year

In [63]:
orders_df_cleaned = (orders_df_cleaned
                     .withColumn('purchase_year', year('order_purchase_timestamp'))
                    )

In [64]:
orders_df_cleaned.show(3)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+---------------------+--------------+-----------------+-------------+------------+-----------------+--------------+-------------------+-------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|approval_time|carrier_handling_time|delivery_delay|delivery_duration|shipping_time|purchase_day|purchase_day_name|purchase_month|purchase_month_name|purchase_year|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+---------------------+--------------+-----------------+-------------+------------+------

### Advanced Transformation

#### Excluding Outliers in Order Item

In [65]:
quantiles = order_items_df_cleaned.approxQuantile('price', [0.01,0.99],0.0)
low_cutOff, high_cutOff = quantiles

In [66]:
low_cutOff, high_cutOff

(9.99, 890.0)

In [67]:
order_items_df_cleaned = order_items_df_cleaned.filter(
    (col("price") >= low_cutOff) &
    (col("price") <= high_cutOff)
)

#### Creating Product Size Category Based on Weight Distribution

In [68]:
product_weight_quartiles = products_df_cleaned.approxQuantile('product_weight_g',[0.20, 0.40, 0.60, 0.80], 0.00)

q1, q2, q3, q4 = product_weight_quartiles

In [69]:
products_df_cleaned = (products_df_cleaned.withColumn(
                                'product_weight_category', 
                                    when(col('product_weight_g') <= q1, 'Very Light')
                                    .when(col('product_weight_g') <= q2, 'Light')
                                    .when(col('product_weight_g') <= q3, 'Medium')
                                    .when(col('product_weight_g') <= q4, 'Heavy')
                                    .otherwise('Very Heavy')
                            )
                      )


In [70]:
products_df_cleaned.show()

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-----------------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_weight_category|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-----------------------+
|418b40c9e83b7e5ac...|   eletrodomesticos_2|               47.0|                     796.0|               3.0|          7300.0|             82.0|             10.0|            54.0|             Very Heavy|
|6803077179d248894...|          eletronicos|               48.0|                     287.0|               1.0|           100.0|             23.0|              8.0|            18.0|

### Storing Cleaned Datasets

In [71]:
customers_df_cleaned.coalesce(1).write.mode('overwrite').parquet('/user/Marho/project/olist/processed/customers')
orders_df_cleaned.coalesce(1).write.mode('overwrite').parquet('/user/Marho/project/olist/processed/orders')
order_items_df_cleaned.coalesce(1).write.mode('overwrite').parquet('/user/Marho/project/olist/processed/orderItems')
products_df_cleaned.coalesce(1).write.mode('overwrite').parquet('/user/Marho/project/olist/processed/products')
paymentDf_cleaned.coalesce(1).write.mode('overwrite').parquet('/user/Marho/project/olist/processed/payments')
geolocation_df_cleaned.coalesce(1).write.mode('overwrite').parquet('/user/Marho/project/olist/processed/geolocations')
sellers_df_cleaned.coalesce(1).write.mode('overwrite').parquet('/user/Marho/project/olist/processed/sellers')
review_cleaned_df.coalesce(1).write.mode('overwrite').parquet('/user/Marho/project/olist/processed/reviews')
products_category_df_cleaned.coalesce(1).write.mode('overwrite').parquet('/user/Marho/project/olist/processed/categories_transalation')

In [72]:
spark.stop()